# LAB 4 - Phân tích dữ liệu thương mại với Spark DataFrame

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, count, avg, sum

In [2]:
spark = SparkSession.builder.appName("Lab4_Ecommerce_Analysis").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 17:42:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Muc 1 - Doc du lieu CSV va tu dong suy ra kieu du lieu
Muc tieu: Nap 5 bang du lieu (Orders, Order_Items, Products, Customer_List, Order_Reviews) bang Spark DataFrame voi inferSchema=True.

In [5]:
orders_df = spark.read.csv('file:///home/vngbthang/IE212.Q21.1_BigDataLAB/LAB4/Orders.csv', header=True, inferSchema=True, sep=';')
order_items_df = spark.read.csv('file:///home/vngbthang/IE212.Q21.1_BigDataLAB/LAB4/Order_Items.csv', header=True, inferSchema=True, sep=';')
products_df = spark.read.csv('file:///home/vngbthang/IE212.Q21.1_BigDataLAB/LAB4/Products.csv', header=True, inferSchema=True, sep=';')
customers_df = spark.read.csv('file:///home/vngbthang/IE212.Q21.1_BigDataLAB/LAB4/Customer_List.csv', header=True, inferSchema=True, sep=';')
reviews_df = spark.read.csv('file:///home/vngbthang/IE212.Q21.1_BigDataLAB/LAB4/Order_Reviews.csv', header=True, inferSchema=True, sep=';')

## Muc 2 - Thong ke tong quan
Muc tieu: Tinh tong so don hang, so luong khach hang duy nhat va so luong nguoi ban duy nhat.

In [6]:
total_orders = orders_df.count()
total_customers = customers_df.select('Customer_Trx_ID').distinct().count()
total_sellers = order_items_df.select('Seller_ID').distinct().count()

print(f'Tổng số đơn hàng: {total_orders}')
print(f'Số lượng khách hàng duy nhất: {total_customers}')
print(f'Số lượng người bán duy nhất: {total_sellers}')

Tổng số đơn hàng: 99441
Số lượng khách hàng duy nhất: 99442
Số lượng người bán duy nhất: 3095


## Muc 3 - So luong don hang theo quoc gia
Muc tieu: Join don hang voi thong tin khach hang, nhom theo quoc gia va sap xep giam dan theo tong so don.

In [7]:
orders_by_country = orders_df.join(customers_df, 'Customer_Trx_ID', 'inner') \
    .groupBy('Customer_Country') \
    .agg(count('Order_ID').alias('Total_Orders')) \
    .orderBy(col('Total_Orders').desc())

print("Số lượng đơn hàng theo quốc gia:")
orders_by_country.show(10)

Số lượng đơn hàng theo quốc gia:
+----------------+------------+
|Customer_Country|Total_Orders|
+----------------+------------+
|         Germany|       41754|
|          France|       12848|
|     Netherlands|       11629|
|         Belgium|        5464|
|         Austria|        5043|
|     Switzerland|        3640|
|  United Kingdom|        3382|
|          Poland|        2139|
|         Czechia|        2034|
|           Italy|        2025|
+----------------+------------+
only showing top 10 rows


## Muc 4 - Phan tich don hang theo nam va thang
Muc tieu: Gom nhom theo nam/thang dat hang, sap xep nam tang dan va thang giam dan.

In [8]:
orders_by_time = orders_df.withColumn('Year', year('Order_Purchase_Timestamp')) \
    .withColumn('Month', month('Order_Purchase_Timestamp')) \
    .groupBy('Year', 'Month') \
    .agg(count('Order_ID').alias('Total_Orders')) \
    .orderBy(col('Year').asc(), col('Month').desc())

print("Số lượng đơn hàng theo năm (tăng dần) và tháng (giảm dần):")
orders_by_time.show(15)

Số lượng đơn hàng theo năm (tăng dần) và tháng (giảm dần):
+----+-----+------------+
|Year|Month|Total_Orders|
+----+-----+------------+
|2022|   12|           1|
|2022|   10|         324|
|2022|    9|           4|
|2023|   12|        5673|
|2023|   11|        7544|
|2023|   10|        4631|
|2023|    9|        4285|
|2023|    8|        4331|
|2023|    7|        4026|
|2023|    6|        3245|
|2023|    5|        3700|
|2023|    4|        2404|
|2023|    3|        2682|
|2023|    2|        1780|
|2023|    1|         800|
+----+-----+------------+
only showing top 15 rows


## Muc 5 - Thong ke danh gia va xu ly ngoai le
Muc tieu: Lam sach cot Review_Score (loai NULL va gia tri khong hop le), sau do tinh diem trung binh va so luong theo tung muc diem 1-5.

In [11]:
from pyspark.sql.functions import expr

# Ép kiểu giá trị không phải số thành NULL, rồi lọc chỉ giữ điểm 1-5
cleaned_reviews = reviews_df.withColumn('Review_Score_Num', expr('try_cast(Review_Score as double)')) \
    .filter(col('Review_Score_Num').isNotNull() & col('Review_Score_Num').between(1, 5))

# Điểm đánh giá trung bình tổng thể
avg_score = cleaned_reviews.select(avg('Review_Score_Num')).collect()[0][0]
print(f'Điểm đánh giá trung bình tổng thể: {avg_score:.2f}\n')

# Số lượng đánh giá theo từng mức độ
reviews_by_score = cleaned_reviews.groupBy('Review_Score_Num') \
    .agg(count('Review_ID').alias('Total_Reviews')) \
    .orderBy('Review_Score_Num')

print("Số lượng đánh giá theo từng mức điểm:")
reviews_by_score.show()

Điểm đánh giá trung bình tổng thể: 4.09

Số lượng đánh giá theo từng mức điểm:
+----------------+-------------+
|Review_Score_Num|Total_Reviews|
+----------------+-------------+
|             1.0|        11424|
|             2.0|         3151|
|             3.0|         8179|
|             4.0|        19141|
|             5.0|        57328|
+----------------+-------------+



## Muc 6 (Lua chon) - Doanh thu nam 2024 theo danh muc
Muc tieu: Tinh doanh thu = Price + Freight_Value, loc don hang nam 2024 va tong hop theo Product_Category_Name.

In [12]:
# Doanh thu = Giá sản phẩm (Price) + Phí vận chuyển (Freight_Value)
revenue_2024 = orders_df.filter(year('Order_Purchase_Timestamp') == 2024) \
    .join(order_items_df, 'Order_ID') \
    .join(products_df, 'Product_ID') \
    .withColumn('Revenue', col('Price') + col('Freight_Value')) \
    .groupBy('Product_Category_Name') \
    .agg(sum('Revenue').alias('Total_Revenue')) \
    .orderBy(col('Total_Revenue').desc())

print("Doanh thu năm 2024 theo danh mục sản phẩm (Top 10):")
revenue_2024.show(10)

Doanh thu năm 2024 theo danh mục sản phẩm (Top 10):
+---------------------+------------------+
|Product_Category_Name|     Total_Revenue|
+---------------------+------------------+
|        Health_Beauty| 885191.1200000007|
|        Watches_Gifts| 771986.7499999991|
|       Bed_Bath_Table| 650794.6999999994|
|       Sports_Leisure| 621999.3399999996|
| Computers_Accesso...| 594771.0400000003|
|           Housewares| 491576.9600000005|
|      Furniture_Decor|476466.12999999983|
|                 Auto|404210.56999999995|
|                 Baby|299052.56000000006|
|           Cool_Stuff|273910.05000000005|
+---------------------+------------------+
only showing top 10 rows


## Muc 7 (Lua chon) - San pham ban chay va diem danh gia trung binh
Muc tieu: Tim san pham co so luong ban cao, dong thoi tinh diem danh gia trung binh tu du lieu da lam sach.

In [14]:
top_products = order_items_df.join(
    cleaned_reviews.select('Order_ID', 'Review_Score_Num'),
    'Order_ID',
    'left'
) \
    .groupBy('Product_ID') \
    .agg(
        count('Order_Item_ID').alias('Total_Sold'),
        avg('Review_Score_Num').alias('Average_Review_Score')
    ) \
    .orderBy(col('Total_Sold').desc())

print("Top 10 sản phẩm bán chạy nhất và điểm đánh giá trung bình:")
top_products.show(10)

Top 10 sản phẩm bán chạy nhất và điểm đánh giá trung bình:
+--------------------+----------+--------------------+
|          Product_ID|Total_Sold|Average_Review_Score|
+--------------------+----------+--------------------+
|aca2eb7d00ea1a7b8...|       527|   4.019083969465649|
|99a4788cb24856965...|       491|  3.8983402489626555|
|422879e10f4668299...|       487|  3.9465020576131686|
|389d119b48cf3043d...|       392|   4.117647058823529|
|368c6c730842d7801...|       391|   3.922680412371134|
|53759a2ecddad2bb8...|       375|   3.868632707774799|
|d1c427060a0f73f6b...|       343|   4.194117647058824|
|53b36df67ebb7c415...|       323|            4.190625|
|154e7e31ebfa09220...|       293|   4.315068493150685|
|3dd2a17168ec895c7...|       274|   4.209558823529412|
+--------------------+----------+--------------------+
only showing top 10 rows
